# Week 2 rollout statistics
Samples 8 answers (temperature 0.9, 512 tokens, HF generate, fp16) for 128 GSM8K train problems with each base model, 0.5B on GPU 0 and 1.5B on GPU 1, and reports how many groups are mixed (some right, some wrong).

In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "math-verify[antlr4_13_2]" 2>&1 | tail -1
!python -m pytest -q tests/ 2>&1 | tail -2
OUT = "/kaggle/working"
jobs = {"base-0.5b": ("Qwen/Qwen2.5-0.5B-Instruct", "0", "16"), "base-1.5b": ("Qwen/Qwen2.5-1.5B-Instruct", "1", "8")}
env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
procs = {}
for name, (model, gpu, bs) in jobs.items():
    cmd = ["python", "-m", "grpo.rollout", "--model", model, "--n-problems", "128", "--batch-size", bs,
           "--dtype", "float16", "--out", f"{OUT}/rollouts/{name}"]
    procs[name] = subprocess.Popen(cmd, env={**env, "CUDA_VISIBLE_DEVICES": gpu},
                                   stdout=open(f"{OUT}/{name}.log", "w"), stderr=subprocess.STDOUT)
for name, p in procs.items():
    print(name, "exit code", p.wait())
import json
for name in jobs:
    path = f"{OUT}/rollouts/{name}/summary.json"
    print(name, json.dumps(json.load(open(path))["summary"], indent=1) if os.path.exists(path) else open(f"{OUT}/{name}.log").read()[-3000:])
